#  LeetCode Prep Assistant — Powered by Claude

A three-mode LeetCode study assistant that uses **Claude Haiku** (`claude-haiku-4.5`) and the LeetCode GraphQL API to give you:

| Mode | Trigger | What happens |
|------|---------|-------------|
| `daily` | Morning run | Spaced-repetition reviews due + recommended next new problem |
| `doubt` | You have a question | Claude answers in context of your full history |
| `optimize` | You solved brute-force, want optimal | Verify AC → hints or full solution |

**State** is stored locally in `study_state.json` (gitignored). No data leaves your machine except API calls to Anthropic and LeetCode.

**Setup:**
1. Install dependencies (next cell).
2. Create a `.env` file with `ANTHROPIC_API_KEY`, `LEETCODE_SESSION`, and `LEETCODE_USERNAME`.
3. Edit the **Mode Selector** cell below, then run all cells.


## Install Dependencies

Uses `uv` (the repo-standard package manager). Run once per environment.

In [ ]:
# Install required packages via uv (repo-standard package manager)
import subprocess

packages = ["anthropic", "requests", "python-dotenv"]
subprocess.check_call(
    ["uv", "pip", "install", "--quiet", *packages],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print("Dependencies installed:", ", ".join(packages))

## Configuration — Mode Selector

**Edit this cell every session.** Set `MODE` to one of `"daily"`, `"doubt"`, or `"optimize"`, then fill the corresponding variable.

- `daily` — no extra fields needed.
- `doubt` — fill `DOUBT_QUESTION`.
- `optimize` — fill `OPTIMIZE_SLUG` and optionally `OPTIMIZE_PREFER`.

In [ ]:
# -- MODE SELECTOR ------------------------------------------
MODE = "daily"  # "daily" | "doubt" | "optimize"

# Fill only the field for the mode you selected:
DOUBT_QUESTION = ""  # e.g. "Why does sliding window work here?"
OPTIMIZE_SLUG = ""  # e.g. "two-sum" (LeetCode problem slug)
OPTIMIZE_PREFER = "hints"  # "hints" | "solution"
# -----------------------------------------------------------

## Setup — Imports and Initialization

Loads environment variables, initialises the Anthropic client, and defines all global constants.

> **Required `.env` keys:**
> - `ANTHROPIC_API_KEY` — from [console.anthropic.com](https://console.anthropic.com)
> - `LEETCODE_SESSION` — cookie from your browser while logged in to LeetCode
> - `LEETCODE_USERNAME` — your LeetCode username

In [ ]:
import hashlib
import json
import os
import time
from datetime import datetime
from pathlib import Path

import anthropic
import requests
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
MODEL_NAME = "claude-haiku-4-5"
STATE_FILE = Path("study_state.json")
REVIEW_INTERVALS_DAYS = [1, 3, 7, 14, 30, 90]
WEAK_THRESHOLD = 4

print(f"Anthropic client initialised. Model: {MODEL_NAME}")

## State Management

`study_state.json` is the **single persistence file** for this notebook. It stores:
- `recommendation_cache` — keyed by SHA-256 of your AC problem titles, avoids redundant Claude calls
- `review_schedule` — per-problem spaced repetition schedule
- `revisit_queue` — problems flagged as weak (auto or manual)
- `doubt_history` — log of all doubt Q&A sessions

In [ ]:
def load_state() -> dict:
    """Load study_state.json, returning a valid default if the file doesn't exist."""
    default = {
        "version": "1.1",
        "recommendation_cache": {},
        "review_schedule": {},
        "revisit_queue": {},
        "doubt_history": [],
    }
    if STATE_FILE.exists():
        return json.loads(STATE_FILE.read_text())
    return default


def save_state(state: dict) -> None:
    """Persist state dict to study_state.json (gitignored)."""
    STATE_FILE.write_text(json.dumps(state, indent=2))


print("State helpers defined.")

### Spaced Repetition Helpers

Implements the review schedule using intervals `[1, 3, 7, 14, 30, 90]` days. Each successful review increments `review_count`, extending the next interval.

In [ ]:
def next_review_ts(review_count: int, solved_ts: int) -> int:
    """Compute the Unix timestamp for the next review based on the review count."""
    idx = min(review_count, len(REVIEW_INTERVALS_DAYS) - 1)
    return int(solved_ts) + REVIEW_INTERVALS_DAYS[idx] * 86400


def get_due_reviews(state: dict) -> list:
    """Return all problems whose next_review_ts is <= now."""
    now = int(datetime.now().timestamp())
    due = []
    for slug, info in state["review_schedule"].items():
        if info["next_review_ts"] <= now:
            due.append(
                {
                    "slug": slug,
                    "title": slug.replace("-", " ").title(),
                    "reason": (
                        "Manually flagged for review"
                        if info.get("manually_flagged")
                        else "Spaced repetition due"
                    ),
                }
            )
    return due


def update_review_after_solve(state: dict, slug: str, solved_ts: int) -> dict:
    """Register a newly solved problem in the review schedule."""
    existing = state["review_schedule"].get(slug, {"review_count": 0})
    count = existing.get("review_count", 0)
    state["review_schedule"][slug] = {
        "last_solved_ts": solved_ts,
        "next_review_ts": next_review_ts(count, solved_ts),
        "review_count": count,
        "manually_flagged": existing.get("manually_flagged", False),
    }
    return state


def mark_reviewed(state: dict, slug: str) -> dict:
    """Increment review_count and push the next review date forward."""
    if slug in state["review_schedule"]:
        info = state["review_schedule"][slug]
        new_count = info["review_count"] + 1
        state["review_schedule"][slug]["review_count"] = new_count
        state["review_schedule"][slug]["next_review_ts"] = next_review_ts(
            new_count, int(datetime.now().timestamp())
        )
    return state


print("Spaced repetition helpers defined.")
print(f"   Review intervals (days): {REVIEW_INTERVALS_DAYS}")

### Weak Problem Detection

Scans all submissions and flags problems where the user needed **4 or more attempts** before the first Accepted submission. These are added to `revisit_queue` with `reason: "auto"`.

In [ ]:
from collections import defaultdict


def detect_weak_problems(all_submissions: list) -> dict:
    """
    Groups all submissions by problem. Flags any problem where
    attempts_before_ac >= WEAK_THRESHOLD.
    Returns dict keyed by titleSlug.
    """
    grouped: dict = defaultdict(list)
    for s in sorted(all_submissions, key=lambda x: int(x["timestamp"])):
        grouped[s["titleSlug"]].append(s)

    weak = {}
    for slug, subs in grouped.items():
        ac_indices = [i for i, s in enumerate(subs) if s["statusDisplay"] == "Accepted"]
        if ac_indices:
            first_ac = ac_indices[0]  # number of non-AC attempts before first AC
            if first_ac >= WEAK_THRESHOLD:
                weak[slug] = {
                    "reason": "auto",
                    "attempts_before_ac": first_ac,
                    "flagged_ts": int(datetime.now().timestamp()),
                }
    return weak


def submission_hash(ac_submissions: list) -> str:
    """SHA-256 of sorted AC problem titles — used as recommendation cache key."""
    titles = sorted(s["title"] for s in ac_submissions)
    return hashlib.sha256(json.dumps(titles).encode()).hexdigest()


print(f"Weak problem detection defined. Threshold: >= {WEAK_THRESHOLD} attempts before AC.")

## LeetCode GraphQL API

All LeetCode data is fetched from `https://leetcode.com/graphql` using your session cookie. The functions below define:

1. `leetcode_query()` — base request helper
2. `fetch_ac_submissions()` — recent accepted submissions
3. `fetch_all_submissions()` — all recent submissions (for weak problem detection)
4. `fetch_tag_stats()` — per-topic solve counts
5. `verify_ac_for_problem()` — check AC exists for a specific problem (optimize mode)

> **Note:** `recentAcSubmissionList` caps at 20 entries regardless of `limit`. `recentSubmissionList` returns up to 100. Code content requires LeetCode Premium — this notebook never requests it.

In [ ]:
LEETCODE_GRAPHQL_URL = "https://leetcode.com/graphql"


def leetcode_query(query: str, variables: dict, session_cookie: str) -> dict:
    """Base GraphQL request to LeetCode. Raises on HTTP or GraphQL errors."""
    response = requests.post(
        LEETCODE_GRAPHQL_URL,
        json={"query": query, "variables": variables},
        headers={
            "Content-Type": "application/json",
            "Cookie": f"LEETCODE_SESSION={session_cookie}",
            "Referer": "https://leetcode.com",
            "User-Agent": "Mozilla/5.0",
        },
        timeout=10,
    )
    response.raise_for_status()
    data = response.json()
    if "errors" in data:
        raise ValueError(f"GraphQL error: {data['errors']}")
    return data


print("Base GraphQL helper defined.")

### Fetch Accepted Submissions

Queries `recentAcSubmissionList` for up to 50 accepted submissions. Note: LeetCode caps this at 20 entries regardless of the `limit` variable.

In [ ]:
AC_SUBMISSIONS_QUERY = """
query recentAcSubmissions($username: String!, $limit: Int!) {
  recentAcSubmissionList(username: $username, limit: $limit) {
    id
    title
    titleSlug
    timestamp
    lang
  }
}
"""


def fetch_ac_submissions(username: str, session_cookie: str) -> list:
    """Fetch recent accepted submissions for the given username."""
    data = leetcode_query(
        AC_SUBMISSIONS_QUERY,
        {"username": username, "limit": 50},
        session_cookie,
    )
    return data.get("data", {}).get("recentAcSubmissionList", [])


print("fetch_ac_submissions() defined.")

### Fetch All Recent Submissions

Queries `recentSubmissionList` including wrong answers, TLEs, etc. Used by `detect_weak_problems()` to identify struggles.

In [ ]:
ALL_SUBMISSIONS_QUERY = """
query recentSubmissions($username: String!, $limit: Int!) {
  recentSubmissionList(username: $username, limit: $limit) {
    id
    title
    titleSlug
    statusDisplay
    timestamp
    lang
  }
}
"""


def fetch_all_submissions(username: str, session_cookie: str) -> list:
    """Fetch all recent submissions (AC + non-AC) for weak-problem detection."""
    data = leetcode_query(
        ALL_SUBMISSIONS_QUERY,
        {"username": username, "limit": 100},
        session_cookie,
    )
    return data.get("data", {}).get("recentSubmissionList", [])


print("fetch_all_submissions() defined.")

### Fetch Tag Statistics

Queries per-topic solve counts across fundamental, intermediate, and advanced tiers. Used by Claude to identify strongest and weakest topics.

In [ ]:
TAG_STATS_QUERY = """
query userTagStats($username: String!) {
  matchedUser(username: $username) {
    tagProblemCounts {
      advanced   { tagName problemsSolved }
      intermediate { tagName problemsSolved }
      fundamental  { tagName problemsSolved }
    }
    submitStats {
      acSubmissionNum { difficulty count }
    }
  }
}
"""


def fetch_tag_stats(username: str, session_cookie: str) -> dict:
    """Fetch per-topic solve counts. Returns a dict with keys: fundamental, intermediate, advanced."""
    try:
        data = leetcode_query(TAG_STATS_QUERY, {"username": username}, session_cookie)
        counts = data.get("data", {}).get("matchedUser", {}).get("tagProblemCounts", {})
        return counts if counts else {}
    except Exception as exc:
        print(f"WARNING: Could not fetch tag stats: {exc}")
        return {}


print("fetch_tag_stats() defined.")

### Verify AC for a Specific Problem (Optimize Mode)

Before providing optimization hints, we verify the user actually has an accepted submission. Returns:
- `True` — AC found
- `False` — no AC found
- `None` — verification failed (premium wall, network error, etc.) → proceed with a warning

In [ ]:
PROBLEM_SUBMISSIONS_QUERY = """
query problemSubmissions($slug: String!, $offset: Int!, $limit: Int!) {
  questionSubmissionList(questionSlug: $slug, offset: $offset, limit: $limit) {
    submissions {
      id
      statusDisplay
      runtime
      timestamp
      lang
    }
  }
}
"""


def verify_ac(slug: str, session_cookie: str) -> bool | None:
    """
    Returns True if an accepted submission exists for the problem,
    False if no AC exists, or None if verification could not be completed
    (e.g., premium wall, network error, missing session cookie).
    """
    if not session_cookie:
        return None
    try:
        data = leetcode_query(
            PROBLEM_SUBMISSIONS_QUERY,
            {"slug": slug, "offset": 0, "limit": 10},
            session_cookie,
        )
        subs = data.get("data", {}).get("questionSubmissionList", {}).get("submissions", [])
        if subs is None:
            return None
        return any(s["statusDisplay"] == "Accepted" for s in subs)
    except Exception:
        return None  # premium wall or network error — caller handles None


print("verify_ac() defined. Returns True | False | None.")

## Claude Tools Definition

Defines the `provide_daily_plan` tool schema used in daily mode. Claude is forced to call this tool via `tool_choice={"type": "tool", "name": "provide_daily_plan"}`, guaranteeing structured JSON output.

In [ ]:
daily_recommendation_tool = {
    "name": "provide_daily_plan",
    "description": "Structured daily LeetCode study plan.",
    "input_schema": {
        "type": "object",
        "properties": {
            "reviews_due": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "title": {"type": "string"},
                        "slug": {"type": "string"},
                        "reason": {"type": "string"},
                    },
                },
                "description": "Problems due for spaced repetition review today",
            },
            "revisit_flagged": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Weak problems from revisit_queue to re-attempt today",
            },
            "next_new_problem": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "difficulty": {
                        "type": "string",
                        "enum": ["Easy", "Medium", "Hard"],
                    },
                    "pattern": {"type": "string"},
                    "leetcode_url": {"type": "string"},
                },
                "required": ["title", "difficulty", "pattern", "leetcode_url"],
            },
            "strongest_topics": {"type": "array", "items": {"type": "string"}},
            "weakest_topics": {"type": "array", "items": {"type": "string"}},
            "reasoning": {"type": "string"},
        },
        "required": [
            "reviews_due",
            "revisit_flagged",
            "next_new_problem",
            "strongest_topics",
            "weakest_topics",
            "reasoning",
        ],
    },
}

print("provide_daily_plan tool schema defined.")
print(f"   Required fields: {daily_recommendation_tool['input_schema']['required']}")

## Mode: Daily

The daily mode does three things:
1. **`build_daily_context()`** — assembles due reviews, revisit flags, and submission summaries.
2. **`get_daily_plan()`** — calls Claude (`temperature=0`) with `tool_choice` forced to `provide_daily_plan`. Caches by submission hash to avoid redundant API calls.
3. **`display_daily_plan()`** — pretty-prints the structured plan.

In [ ]:
def build_daily_context(
    ac_submissions: list,
    tag_stats: dict,
    state: dict,
) -> dict:
    """
    Assembles all context needed for the daily plan:
    - due_reviews: problems whose next_review_ts <= now
    - revisit_slugs: all slugs currently in revisit_queue
    - history_text: formatted solve history
    - tag_lines: formatted topic solve counts
    """
    due_reviews = get_due_reviews(state)
    revisit_slugs = list(state["revisit_queue"].keys())

    history_text = (
        "\n".join(f"- {s['title']} ({s['lang']})" for s in ac_submissions)
        or "No solved problems yet."
    )

    tag_lines = (
        "\n".join(
            f"- {t['tagName']}: {t['problemsSolved']} solved"
            for tier in ["fundamental", "intermediate", "advanced"]
            for t in tag_stats.get(tier, [])
        )
        or "Tag statistics not available."
    )

    due_text = "\n".join(f"- {r['title']} ({r['reason']})" for r in due_reviews) or "None"
    revisit_text = "\n".join(f"- {slug}" for slug in revisit_slugs) or "None"

    return {
        "due_reviews": due_reviews,
        "revisit_slugs": revisit_slugs,
        "history_text": history_text,
        "tag_lines": tag_lines,
        "due_text": due_text,
        "revisit_text": revisit_text,
    }


print("build_daily_context() defined.")

### Get Daily Plan from Claude

Calls Claude with `temperature=0` and `tool_choice` locked to `provide_daily_plan`. The response is cached by submission hash so that re-running without new solves returns the same plan instantly.

In [ ]:
def get_daily_plan(
    ac_submissions: list,
    tag_stats: dict,
    due_reviews: list,
    revisit_slugs: list,
    state: dict,
) -> dict:
    """
    Calls Claude (temperature=0) with tool_choice forced to provide_daily_plan.
    Caches the result by SHA-256 of AC submission titles.
    """
    cache_key = submission_hash(ac_submissions)
    cache = state["recommendation_cache"]

    if cache_key in cache:
        print("Cache hit — recommendation unchanged since last solve.")
        return cache[cache_key]

    print("New submission state — calling Claude for updated plan...")

    ctx = build_daily_context(ac_submissions, tag_stats, state)

    prompt = f"""You are a LeetCode study mentor. Analyze this user's history and produce today's study plan.

## Solved problems:
{ctx["history_text"]}

## Topic solve counts:
{ctx["tag_lines"]}

## Problems due for spaced repetition review today:
{ctx["due_text"]}

## Weak problems flagged for revisit:
{ctx["revisit_text"]}

Produce a structured daily study plan using the provide_daily_plan tool."""

    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=1024,
        temperature=0,  # deterministic — same input always produces same plan
        tools=[daily_recommendation_tool],
        tool_choice={"type": "tool", "name": "provide_daily_plan"},
        messages=[{"role": "user", "content": prompt}],
    )

    plan = next(
        b.input for b in response.content if b.type == "tool_use" and b.name == "provide_daily_plan"
    )

    # Inject the locally-computed due reviews so display is always accurate
    plan["reviews_due"] = due_reviews

    cache[cache_key] = plan
    save_state(state)
    return plan


print("get_daily_plan() defined. temperature=0, tool_choice=provide_daily_plan.")

### Display Daily Plan

Pretty-prints the structured plan returned by Claude.

In [ ]:
def display_daily_plan(plan: dict) -> None:
    """Pretty-print the structured daily plan."""
    print("\nTODAY'S LEETCODE PLAN")
    print("=" * 55)

    if plan.get("reviews_due"):
        print("\nSpaced repetition reviews due:")
        for r in plan["reviews_due"]:
            print(f"   • {r['title']} — {r['reason']}")
    else:
        print("\nNo reviews due today.")

    if plan.get("revisit_flagged"):
        print("\nWARNING:  Weak problems to revisit:")
        for slug in plan["revisit_flagged"]:
            print(f"   • {slug.replace('-', ' ').title()}")

    if plan.get("strongest_topics"):
        print(f"\nStrongest topics: {', '.join(plan['strongest_topics'])}")
    if plan.get("weakest_topics"):
        print(f"WARNING:  Weakest topics:  {', '.join(plan['weakest_topics'])}")

    np = plan.get("next_new_problem", {})
    if np:
        print(f"\nNext new problem: {np.get('title', 'N/A')}")
        print(f"    Difficulty:       {np.get('difficulty', 'N/A')}")
        print(f"    Pattern:          {np.get('pattern', 'N/A')}")
        print(f"    Link:             {np.get('leetcode_url', 'N/A')}")

    if plan.get("reasoning"):
        print(f"\nReasoning:\n   {plan['reasoning']}")


print("display_daily_plan() defined.")

## Mode: Doubt

Doubt mode answers your LeetCode questions using your **full solve history as context**. Claude uses `temperature=0.3` here — answers are conversational, slight phrasing variation is acceptable, but correctness is paramount. Every Q&A pair is logged to `doubt_history` in `study_state.json`.

In [ ]:
def answer_doubt(question: str, ac_submissions: list, state: dict) -> str:
    """
    Calls Claude (temperature=0.3) with the user's solve history as context.
    Stores the Q&A pair in state['doubt_history'].
    """
    history_text = "\n".join(f"- {s['title']}" for s in ac_submissions) or "No solved problems yet."

    prompt = f"""You are a LeetCode tutor. Answer the user's question clearly and concisely.
Use their solve history as context — tailor the explanation to their level.

## User's solved problems:
{history_text}

## User's question:
{question}"""

    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=1024,
        temperature=0.3,  # conversational — slight variation is fine
        messages=[{"role": "user", "content": prompt}],
    )
    answer = response.content[0].text

    state["doubt_history"].append(
        {
            "timestamp": int(datetime.now().timestamp()),
            "question": question,
            "answer": answer,
        }
    )
    save_state(state)
    return answer


print("answer_doubt() defined. temperature=0.3.")

### Display Doubt Answer

In [ ]:
def display_doubt_answer(question: str, answer: str) -> None:
    """Pretty-print the doubt Q&A."""
    print("\nYour question:")
    print(f"    {question}")
    print("\nAnswer:")
    print(answer)


print("display_doubt_answer() defined.")

## Mode: Optimize

Optimize mode helps you move from brute force to the optimal solution. It:
1. **Verifies** you have an accepted submission (`verify_ac()`).
2. **Provides hints** (`OPTIMIZE_PREFER="hints"`) — 3 progressive hints, increasingly specific.
3. **Provides solution** (`OPTIMIZE_PREFER="solution"`) — complexity analysis, key insight, pseudocode, and a follow-up problem.

Both paths use `temperature=0` for deterministic, repeatable guidance.

In [ ]:
def get_optimize_hints(slug: str, ac_submissions: list) -> str:
    """
    Provides 3 progressive hints for the given problem (temperature=0).
    Hint 1: high-level direction. Hint 2: data structure/technique. Hint 3: specific insight.
    """
    history_text = "\n".join(f"- {s['title']}" for s in ac_submissions) or "No solved problems yet."

    prompt = f"""You are a LeetCode mentor helping a user optimize their solution.
Problem: {slug.replace("-", " ").title()} (slug: {slug})
The user has already solved this problem (brute force). They want optimization hints.

## User's solve history for context:
{history_text}

Provide exactly 3 progressive hints for optimizing this problem.
Hint 1: very high-level (the general direction, no specifics).
Hint 2: the key data structure or technique to use.
Hint 3: the specific insight that makes it work, without writing code.
Label them clearly: Hint 1, Hint 2, Hint 3.
Never reveal the full approach outright — prompt the user to think between hints."""

    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=1024,
        temperature=0,  # deterministic hints
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text


print("get_optimize_hints() defined. temperature=0.")

### Get Optimize Solution

When `OPTIMIZE_PREFER="solution"`, provides the full optimal approach: complexity comparison, key insight, pseudocode (not full code), and a follow-up problem.

In [ ]:
def get_optimize_solution(slug: str, ac_submissions: list) -> str:
    """
    Provides the full optimal approach for the given problem (temperature=0):
    brute-force vs optimal complexity, key insight, pseudocode, follow-up problem.
    """
    history_text = "\n".join(f"- {s['title']}" for s in ac_submissions) or "No solved problems yet."

    prompt = f"""You are a LeetCode mentor helping a user optimize their solution.
Problem: {slug.replace("-", " ").title()} (slug: {slug})
The user has already solved this problem (brute force). They want the full optimal approach.

## User's solve history for context:
{history_text}

Provide the full optimization approach:
1. Briefly state the brute force time and space complexity.
2. State the optimal approach's time and space complexity.
3. Explain the key insight that unlocks the optimization (2-3 sentences).
4. Pseudocode of the optimal approach (not full code — keeps the learning active).
5. One follow-up problem to consolidate this pattern (title + LeetCode URL)."""

    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=1024,
        temperature=0,  # deterministic solution
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text


print("get_optimize_solution() defined. temperature=0.")

### Display Optimize Output

In [ ]:
def display_optimize_output(slug: str, prefer: str, output: str) -> None:
    """Pretty-print the optimization hints or solution."""
    label = " HINTS" if prefer == "hints" else " OPTIMAL APPROACH"
    print(f"\n{label} — {slug.replace('-', ' ').title()}\n")
    print(output)


print("display_optimize_output() defined.")

## Mock Data (Offline Testing)

When `LEETCODE_SESSION` is not set, the notebook falls back to this mock dataset. This lets you explore all three modes without a live LeetCode cookie and is used in the example outputs below.

In [ ]:
MOCK_AC_SUBMISSIONS = [
    {
        "id": "1001",
        "title": "Two Sum",
        "titleSlug": "two-sum",
        "lang": "c",
        "statusDisplay": "Accepted",
        "timestamp": "1719000000",
    },
    {
        "id": "1002",
        "title": "Best Time to Buy and Sell Stock",
        "titleSlug": "best-time-to-buy-and-sell-stock",
        "lang": "c",
        "statusDisplay": "Accepted",
        "timestamp": "1719100000",
    },
    {
        "id": "1003",
        "title": "Contains Duplicate",
        "titleSlug": "contains-duplicate",
        "lang": "c",
        "statusDisplay": "Accepted",
        "timestamp": "1719200000",
    },
    {
        "id": "1004",
        "title": "Maximum Subarray",
        "titleSlug": "maximum-subarray",
        "lang": "c",
        "statusDisplay": "Accepted",
        "timestamp": "1719300000",
    },
    {
        "id": "1005",
        "title": "House Robber",
        "titleSlug": "house-robber",
        "lang": "c",
        "statusDisplay": "Accepted",
        "timestamp": "1719400000",
    },
    {
        "id": "1006",
        "title": "Climbing Stairs",
        "titleSlug": "climbing-stairs",
        "lang": "c",
        "statusDisplay": "Accepted",
        "timestamp": "1719500000",
    },
    {
        "id": "1007",
        "title": "Valid Parentheses",
        "titleSlug": "valid-parentheses",
        "lang": "c",
        "statusDisplay": "Accepted",
        "timestamp": "1719600000",
    },
    {
        "id": "1008",
        "title": "Merge Two Sorted Lists",
        "titleSlug": "merge-two-sorted-lists",
        "lang": "c",
        "statusDisplay": "Accepted",
        "timestamp": "1719700000",
    },
    {
        "id": "1009",
        "title": "Binary Search",
        "titleSlug": "binary-search",
        "lang": "c",
        "statusDisplay": "Accepted",
        "timestamp": "1719800000",
    },
    {
        "id": "1010",
        "title": "Reverse Linked List",
        "titleSlug": "reverse-linked-list",
        "lang": "c",
        "statusDisplay": "Accepted",
        "timestamp": "1719900000",
    },
]

MOCK_ALL_SUBMISSIONS = [
    {
        "id": "1001",
        "title": "Two Sum",
        "titleSlug": "two-sum",
        "statusDisplay": "Accepted",
        "timestamp": "1719000000",
    },
    {
        "id": "900",
        "title": "Two Sum",
        "titleSlug": "two-sum",
        "statusDisplay": "Wrong Answer",
        "timestamp": "1718990000",
    },
    {
        "id": "1010",
        "title": "Reverse Linked List",
        "titleSlug": "reverse-linked-list",
        "statusDisplay": "Accepted",
        "timestamp": "1719900000",
    },
    {
        "id": "1009",
        "title": "Reverse Linked List",
        "titleSlug": "reverse-linked-list",
        "statusDisplay": "Wrong Answer",
        "timestamp": "1719880000",
    },
    {
        "id": "1008",
        "title": "Reverse Linked List",
        "titleSlug": "reverse-linked-list",
        "statusDisplay": "Time Limit Exceeded",
        "timestamp": "1719870000",
    },
    {
        "id": "1007",
        "title": "Reverse Linked List",
        "titleSlug": "reverse-linked-list",
        "statusDisplay": "Wrong Answer",
        "timestamp": "1719860000",
    },
    {
        "id": "1006",
        "title": "Reverse Linked List",
        "titleSlug": "reverse-linked-list",
        "statusDisplay": "Wrong Answer",
        "timestamp": "1719850000",
    },
]

MOCK_TAG_STATS = {
    "fundamental": [
        {"tagName": "Array", "problemsSolved": 4},
        {"tagName": "String", "problemsSolved": 1},
        {"tagName": "Linked List", "problemsSolved": 1},
    ],
    "intermediate": [
        {"tagName": "Dynamic Programming", "problemsSolved": 2},
        {"tagName": "Binary Search", "problemsSolved": 1},
        {"tagName": "Stack", "problemsSolved": 1},
    ],
    "advanced": [],
}

print("Mock data loaded.")
print(
    f"   {len(MOCK_AC_SUBMISSIONS)} mock AC submissions, {len(MOCK_ALL_SUBMISSIONS)} total mock submissions."
)

## Main Execution Router

Reads `LEETCODE_SESSION` from the environment. If missing, falls back to mock data with a clear notice. Then:
1. Syncs new AC solves into the review schedule.
2. Detects and saves weak problems.
3. Routes to the mode selected at the top of the notebook.

In [ ]:
# -- Load credentials from environment -----------------------------------------
USE_MOCK = not bool(os.environ.get("LEETCODE_SESSION"))
SESSION = os.environ.get("LEETCODE_SESSION", "")
USERNAME = os.environ.get("LEETCODE_USERNAME", "")
state = load_state()

# -- Fetch or mock submission data ----------------------------------------------
if USE_MOCK:
    print("NOTE: No LEETCODE_SESSION found — using mock data.\n")
    ac_submissions = MOCK_AC_SUBMISSIONS
    all_submissions = MOCK_ALL_SUBMISSIONS
    tag_stats = MOCK_TAG_STATS
else:
    print(f" Fetching data for LeetCode user: {USERNAME}")
    ac_submissions = fetch_ac_submissions(USERNAME, SESSION)
    time.sleep(0.5)  # rate limit
    all_submissions = fetch_all_submissions(USERNAME, SESSION)
    time.sleep(0.5)  # rate limit
    tag_stats = fetch_tag_stats(USERNAME, SESSION)
    print(
        f"   {len(ac_submissions)} AC submissions, {len(all_submissions)} total submissions fetched."
    )

# -- Sync new AC solves into the review schedule -------------------------------
new_solves = 0
for s in ac_submissions:
    if s["titleSlug"] not in state["review_schedule"]:
        state = update_review_after_solve(state, s["titleSlug"], int(s["timestamp"]))
        new_solves += 1

if new_solves:
    print(f"   {new_solves} new problem(s) added to review schedule.")

# -- Detect and save weak problems ---------------------------------------------
weak = detect_weak_problems(all_submissions)
new_flags = 0
for slug, info in weak.items():
    if slug not in state["revisit_queue"]:
        state["revisit_queue"][slug] = info
        new_flags += 1

if new_flags:
    print(f"   WARNING: {new_flags} problem(s) auto-flagged as weak and added to revisit queue.")

save_state(state)

# -- Route to selected mode -----------------------------------------------------
print(f"\nRunning in MODE = '{MODE}'")
print("-" * 55)

if MODE == "daily":
    due_reviews = get_due_reviews(state)
    revisit_slugs = list(state["revisit_queue"].keys())
    plan = get_daily_plan(ac_submissions, tag_stats, due_reviews, revisit_slugs, state)
    display_daily_plan(plan)

elif MODE == "doubt":
    if not DOUBT_QUESTION.strip():
        print("ERROR:  Set DOUBT_QUESTION before running in doubt mode.")
    else:
        answer = answer_doubt(DOUBT_QUESTION, ac_submissions, state)
        display_doubt_answer(DOUBT_QUESTION, answer)

elif MODE == "optimize":
    if not OPTIMIZE_SLUG.strip():
        print("ERROR:  Set OPTIMIZE_SLUG before running in optimize mode.")
    else:
        print(f" Verifying AC submission for: {OPTIMIZE_SLUG}...")
        verified = verify_ac(OPTIMIZE_SLUG, SESSION)

        if verified is False:
            print(
                "ERROR:  No accepted submission found for this problem yet. "
                "Solve it first, then come back for optimization help."
            )
        else:
            if verified is None:
                print(
                    "WARNING: Could not verify submission (skipping check — premium wall or mock mode)."
                )
            else:
                print(" Accepted submission confirmed.")

            if OPTIMIZE_PREFER == "hints":
                output = get_optimize_hints(OPTIMIZE_SLUG, ac_submissions)
            else:
                output = get_optimize_solution(OPTIMIZE_SLUG, ac_submissions)

            display_optimize_output(OPTIMIZE_SLUG, OPTIMIZE_PREFER, output)

else:
    print(f"ERROR:  Unknown MODE: '{MODE}'. Use 'daily', 'doubt', or 'optimize'.")

## Example Outputs

The cells below show **hardcoded** representative outputs for all three modes, so readers of the cookbook can understand what to expect without running the notebook themselves.

---

### Example: Daily Mode Output

```
New submission state — calling Claude for updated plan...

 TODAY'S LEETCODE PLAN
=======================================================

 Spaced repetition reviews due:
   • Reverse Linked List — Spaced repetition due
   • Two Sum — Spaced repetition due

WARNING:  Weak problems to revisit:
   • Reverse Linked List

 Strongest topics: Array, Dynamic Programming
WARNING:  Weakest topics:  Sliding Window, Graph, Heap

 Next new problem: Longest Substring Without Repeating Characters
    Difficulty:       Medium
    Pattern:          Sliding Window
    Link:             https://leetcode.com/problems/longest-substring-without-repeating-characters/

 Reasoning:
    The user has solid foundations in Arrays and DP, but has not yet
    attempted any Sliding Window problems. This is a critical pattern
    for Medium-difficulty interviews. The recommended problem is the
    canonical entry point for this pattern.
```

---

### Example: Doubt Mode Output

```
 Your question:
    Why does sliding window work for longest substring without repeating characters?

 Answer:
    Great question! You've already seen this idea in Maximum Subarray (Kadane's),
    where you expand or reset a window based on a running condition.

    Sliding window works here because the constraint — "no repeating characters" —
    is monotonic: once a window becomes invalid, shrinking it from the left is
    always sufficient to restore validity. You never need to restart from scratch.

    The key idea:
    - Use a hash map to store the last seen index of each character.
    - `left` pointer marks the start of the current valid window.
    - When you see a duplicate at index `right`, jump `left` to
      `max(left, last_seen[char] + 1)` to skip over it.
    - Track the maximum window size seen.

    This gives O(n) time — each character is visited at most twice.
```

---

### Example: Optimize Mode Output (hints)

```
 Verifying AC submission for: two-sum...
 Accepted submission confirmed.

 HINTS — Two Sum

Hint 1:
Think about what information you need to remember from elements you've already seen.
Can you avoid the nested loop entirely?

Hint 2:
A hash map (dictionary) lets you look up whether a needed complement exists in O(1).
What would you store as the key, and what as the value?

Hint 3:
For each number `x`, you need `target - x`. Store each number's index in the map
as you iterate. Before storing, check if `target - x` is already in the map.
If it is, you've found your pair — return both indices.
```

## Customization Ideas

Here are ways to extend this notebook for your own study workflow:

### Manual Revisit Queue
Add a problem to `revisit_queue` manually to force it into your daily plan:
```python
state = load_state()
state["revisit_queue"]["trapping-rain-water"] = {
    "reason": "manual",
    "attempts_before_ac": None,
    "flagged_ts": int(datetime.now().timestamp()),
}
save_state(state)
```

### Mark a Problem as Reviewed
After completing a spaced repetition review, bump the interval:
```python
state = load_state()
state = mark_reviewed(state, "two-sum")
save_state(state)
```

### View Doubt History
See all your past Q&A sessions:
```python
state = load_state()
for entry in state["doubt_history"]:
    ts = datetime.fromtimestamp(entry["timestamp"]).strftime("%Y-%m-%d %H:%M")
    print(f"[{ts}] Q: {entry['question'][:80]}...")
```

### Adjust the Weak Problem Threshold
Change `WEAK_THRESHOLD` in the constants cell (default: 4 attempts before first AC):
```python
WEAK_THRESHOLD = 3  # flag if 3+ attempts before first AC
```

### Adjust Spaced Repetition Intervals
Modify the `REVIEW_INTERVALS_DAYS` list to suit your schedule:
```python
REVIEW_INTERVALS_DAYS = [1, 2, 5, 10, 21, 60]  # more aggressive
```

### Clear the Recommendation Cache
Force Claude to regenerate a fresh plan (e.g., after changing your study goals):
```python
state = load_state()
state["recommendation_cache"] = {}
save_state(state)
```